In [1]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report
)

In [2]:
ARTIFACTS = Path("../artifacts")

FEATURES = ARTIFACTS / "features"
MODELS = ARTIFACTS / "models"
RESULTS = ARTIFACTS / "model_results"

MODELS.mkdir(exist_ok=True)
RESULTS.mkdir(exist_ok=True)

# Load train and validation data

In [3]:
train = pd.read_parquet(FEATURES / "train_features.parquet")
validation = pd.read_parquet(FEATURES / "validation_features.parquet")

print("Train shape:", train.shape)
print("Validation shape:", validation.shape)

Train shape: (67533, 43)
Validation shape: (14471, 43)


# Separate features and target

In [4]:
X_train = train.drop(columns=["label"])
y_train = (train["label"] == "Late").astype(int)

X_validation = validation.drop(columns=["label"])
y_validation = (validation["label"] == "Late").astype(int)

print("X_train:", X_train.shape)
print("X_validation:", X_validation.shape)

print("\nTraining labels:")
print(y_train.value_counts())

print("\nValidation labels:")
print(y_validation.value_counts())

X_train: (67533, 42)
X_validation: (14471, 42)

Training labels:
label
0    61436
1     6097
Name: count, dtype: int64

Validation labels:
label
0    13698
1      773
Name: count, dtype: int64


# Define evaluation function

In [5]:
def evaluate_model(model, X, y, model_name):
    predictions = model.predict(X)
    probabilities = model.predict_proba(X)[:, 1]

    results = {
        "model": model_name,
        "accuracy": accuracy_score(y, predictions),
        "precision": precision_score(y, predictions, zero_division=0),
        "recall": recall_score(y, predictions, zero_division=0),
        "f1": f1_score(y, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y, probabilities),
        "pr_auc": average_precision_score(y, probabilities)
    }

    return results

# Simple baseline

In [6]:
baseline = DummyClassifier(strategy="most_frequent")

baseline.fit(X_train, y_train)

baseline_results = evaluate_model(
    baseline,
    X_validation,
    y_validation,
    "Majority Baseline"
)

pd.DataFrame([baseline_results])

,model,accuracy,precision,recall,f1,roc_auc,pr_auc
0,Majority Baseline,0.946583,0.0,0.0,0.0,0.5,0.053417


# Logistic Regression

In [7]:
logistic_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

logistic_model.fit(X_train, y_train)

logistic_results = evaluate_model(
    logistic_model,
    X_validation,
    y_validation,
    "Logistic Regression"
)

pd.DataFrame([logistic_results])

,model,accuracy,precision,recall,f1,roc_auc,pr_auc
0,Logistic Regression,0.684887,0.10675,0.664942,0.183966,0.749023,0.137344


# Random Forest

In [8]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_results = evaluate_model(
    rf_model,
    X_validation,
    y_validation,
    "Random Forest"
)

pd.DataFrame([rf_results])

,model,accuracy,precision,recall,f1,roc_auc,pr_auc
0,Random Forest,0.94375,0.098039,0.006468,0.012136,0.641806,0.082023


# Compare models

In [9]:
results = pd.DataFrame([
    baseline_results,
    logistic_results,
    rf_results
])

results = results.sort_values(
    by="f1",
    ascending=False
).reset_index(drop=True)

results

,model,accuracy,precision,recall,f1,roc_auc,pr_auc
0,Logistic Regression,0.684887,0.106750,0.664942,0.183966,0.749023,0.137344
1,Random Forest,0.943750,0.098039,0.006468,0.012136,0.641806,0.082023
2,Majority Baseline,0.946583,0.000000,0.000000,0.000000,0.500000,0.053417


# Tune the Random Forest

In [10]:
param_options = [
    {
        "C": 0.01,
        "class_weight": "balanced"
    },
    {
        "C": 0.1,
        "class_weight": "balanced"
    },
    {
        "C": 1.0,
        "class_weight": "balanced"
    },
    {
        "C": 10.0,
        "class_weight": "balanced"
    }
]


In [11]:
tuning_results = []

for i, params in enumerate(param_options, start=1):

    model = LogisticRegression(
        C=params["C"],
        class_weight=params["class_weight"],
        max_iter=1000,
        random_state=42
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_validation)
    probabilities = model.predict_proba(X_validation)[:, 1]

    result = {
        "model": f"Logistic Regression {i}",
        "C": params["C"],
        "class_weight": params["class_weight"],
        "precision": precision_score(
            y_validation,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_validation,
            predictions,
            zero_division=0
        ),
        "f1": f1_score(
            y_validation,
            predictions,
            zero_division=0
        ),
        "roc_auc": roc_auc_score(
            y_validation,
            probabilities
        ),
        "pr_auc": average_precision_score(
            y_validation,
            probabilities
        )
    }

    tuning_results.append(result)

tuning_results = pd.DataFrame(tuning_results)

tuning_results.sort_values(
    by="f1",
    ascending=False
)

,model,C,class_weight,precision,recall,f1,roc_auc,pr_auc
3,Logistic Regression 4,10.00,balanced,0.107342,0.646831,0.184128,0.744959,0.135274
2,Logistic Regression 3,1.00,balanced,0.106750,0.664942,0.183966,0.749023,0.137344
1,Logistic Regression 2,0.10,balanced,0.105221,0.672704,0.181977,0.750943,0.138004
0,Logistic Regression 1,0.01,balanced,0.104481,0.672704,0.180870,0.748019,0.135136


# Select the best configuration

In [12]:
best_row = tuning_results.sort_values(
    by="f1",
    ascending=False
).iloc[0]

best_C = best_row["C"]

print("Best C:", best_C)
print("Best validation F1:", best_row["f1"])

Best C: 10.0
Best validation F1: 0.18412815319462345


# Train final model

In [13]:
final_model = LogisticRegression(
    C=best_C,
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

final_model.fit(X_train, y_train)

print("Final Logistic Regression model trained.")

Final Logistic Regression model trained.


# Evaluate final model on validation

In [14]:
validation_results = evaluate_model(
    final_model,
    X_validation,
    y_validation,
    "Final Logistic Regression"
)

pd.DataFrame([validation_results])

,model,accuracy,precision,recall,f1,roc_auc,pr_auc
0,Final Logistic Regression,0.693801,0.107342,0.646831,0.184128,0.744959,0.135274


# Classification report

In [15]:
validation_predictions = final_model.predict(X_validation)

print(
    classification_report(
        y_validation,
        validation_predictions,
        target_names=["On Time", "Late"],
        zero_division=0
    )
)

              precision    recall  f1-score   support

     On Time       0.97      0.70      0.81     13698
        Late       0.11      0.65      0.18       773

    accuracy                           0.69     14471
   macro avg       0.54      0.67      0.50     14471
weighted avg       0.93      0.69      0.78     14471



In [16]:
model_path = MODELS / "final_model.joblib"

joblib.dump(final_model, model_path)

print(f"Model saved to: {model_path}")

Model saved to: ../artifacts/models/final_model.joblib


In [17]:
results_summary = pd.concat(
    [
        results,
        tuning_results
    ],
    ignore_index=True
)

results_summary.to_csv(
    RESULTS / "results_summary.csv",
    index=False
)

print("Results saved.")

Results saved.


In [18]:
print("Saved model:")
print(model_path.exists())

print("\nSaved results:")
print((RESULTS / "results_summary.csv").exists())

Saved model:
True

Saved results:
True


# test set

In [19]:
test = pd.read_parquet(
    FEATURES / "test_features.parquet"
)

X_test = test.drop(columns=["label"])
y_test = (test["label"] == "Late").astype(int)

print("Test shape:", test.shape)

Test shape: (14472, 43)


In [20]:
test_results = evaluate_model(
    final_model,
    X_test,
    y_test,
    "Final Logistic Regression - Test"
)

pd.DataFrame([test_results])

,model,accuracy,precision,recall,f1,roc_auc,pr_auc
0,Final Logistic Regression - Test,0.441542,0.094756,0.870428,0.170907,0.673308,0.115551


In [21]:
test_predictions = final_model.predict(X_test)

print(
    classification_report(
        y_test,
        test_predictions,
        target_names=["On Time", "Late"],
        zero_division=0
    )
)

              precision    recall  f1-score   support

     On Time       0.98      0.41      0.58     13515
        Late       0.09      0.87      0.17       957

    accuracy                           0.44     14472
   macro avg       0.54      0.64      0.37     14472
weighted avg       0.92      0.44      0.55     14472



In [22]:
final_test_results = pd.DataFrame([test_results])

final_test_results.to_csv(
    RESULTS / "final_test_results.csv",
    index=False
)

final_test_results

,model,accuracy,precision,recall,f1,roc_auc,pr_auc
0,Final Logistic Regression - Test,0.441542,0.094756,0.870428,0.170907,0.673308,0.115551


In [23]:
print("Notebook 6 completed successfully.")

print("\nFinal model:")
print("Logistic Regression")

print("\nValidation F1:")
print(round(validation_results["f1"], 4))

print("\nTest F1:")
print(round(test_results["f1"], 4))

print("\nTest PR-AUC:")
print(round(test_results["pr_auc"], 4))

print("\nArtifacts:")
print(model_path)
print(RESULTS / "results_summary.csv")
print(RESULTS / "final_test_results.csv")

Notebook 6 completed successfully.

Final model:
Logistic Regression

Validation F1:
0.1841

Test F1:
0.1709

Test PR-AUC:
0.1156

Artifacts:
../artifacts/models/final_model.joblib
../artifacts/model_results/results_summary.csv
../artifacts/model_results/final_test_results.csv


## Conclusion

Several classification approaches were evaluated using metrics appropriate for
the imbalanced target. A majority-class baseline was first established, followed
by Logistic Regression and Random Forest models.

Because Late orders represent a minority class, F1-score for the Late class was
used as the primary model-selection metric, while precision, recall, ROC-AUC, and
PR-AUC were also considered.

Logistic Regression substantially outperformed Random Forest on the validation
set. The Logistic Regression model with `C=10` and `class_weight="balanced"`
achieved the best validation F1-score of 0.1841 and was selected as the final
model.

The final model was then evaluated once on the held-out test set. It achieved an
F1-score of 0.1709, recall of 0.8704, precision of 0.0948, ROC-AUC of 0.6733,
and PR-AUC of 0.1156.

The high recall indicates that the model is able to identify most Late orders.
However, the relatively low precision shows that many orders predicted as Late
are actually On Time. This indicates a trade-off between detecting as many Late
orders as possible and reducing false alarms.

Overall, the model provides a useful baseline for predicting late deliveries,
but further improvement would require additional feature engineering, alternative
models, and potentially decision-threshold optimization.